# DysonianLineCNN — Infer on a real experimental spectrum

Thin wrapper around [`dyson_cnn.infer.predict_real_spectrum`](../dyson_cnn/infer.py).

**Before running**:

1. A trained run exists in `runs/` (use [`01_train_and_eval.ipynb`](01_train_and_eval.ipynb)
   if you do not have one yet).
2. `config/inference.json.runName` points at it.
3. `matlab/PrepareOneSpectrumForCNN.m` has been run to produce
   `<spectrum_basename>_spectrum.csv` in the project folder.

On a fresh Colab runtime the first cell clones this **public** repository
automatically — no keys or secrets needed.

## 1. Environment setup

Idempotent across Google Colab and local Mac.

**On Colab** (`google.colab` module detected):
1. Clone (or pull) the public repo `https://github.com/AndriiUriadov/DysonianLineCNN.git`
   into `/content/DysonianLineCNN` — no SSH key or Colab secret required.
2. `pip install -e . --no-deps` the package in editable mode so imports work
   from any cwd while preserving Colab's curated TensorFlow stack.
3. Mount Google Drive at `/content/drive`.

**On local Mac**: skip the Colab bootstrap; just verify that the notebook is
running from the repository root so `import dyson_cnn` resolves.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # --- Colab-only: public git clone -> pip install -> drive mount ---
    # The repository is public, so no SSH key or Colab secret is needed.
    repo_dir = '/content/DysonianLineCNN'
    if not os.path.isdir(repo_dir):
        get_ipython().system('git clone https://github.com/AndriiUriadov/DysonianLineCNN.git ' + repo_dir)
    else:
        get_ipython().system('cd ' + repo_dir + ' && git pull')

    os.chdir(repo_dir)
    get_ipython().system('pip install -q -e . --no-deps')

    from google.colab import drive
    drive.mount('/content/drive')

# Verify we are at the repo root (works on both Colab and local Mac)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dyson_cnn').is_dir():
    for parent in Path.cwd().parents:
        if (parent / 'dyson_cnn').is_dir():
            REPO_ROOT = parent
            os.chdir(REPO_ROOT)
            break
    else:
        raise RuntimeError(f'Could not find repo root from {Path.cwd()}')

print(f'Environment : {"Colab" if IN_COLAB else "local Mac"}')
print(f'Repo root   : {REPO_ROOT}')

## 2. Load configs and resolve paths

Set `SET_NAME` to match the set used during training (e.g. `"set-1"`).
Leave `None` for legacy single-dataset mode.

In [ ]:
from dyson_cnn.config import load_paths, load_inference_cfg

SET_NAME = "set-1"  # e.g. "set-1", "set-2", ... or None for legacy mode

paths = load_paths(REPO_ROOT / 'config', set_name=SET_NAME)
inference_cfg = load_inference_cfg(REPO_ROOT / 'config')

run_name = inference_cfg['runName']
spectrum_basename = inference_cfg['spectrum_basename']

# Per-set paths: run_dir and artifacts under set subdirectory
base_dir = Path(paths.get('set_project_dir', paths['project_dir']))
runs_dir = Path(paths.get('set_runs_dir', paths['runs_dir']))

run_dir = runs_dir / run_name
spectrum_csv = base_dir / f'{spectrum_basename}_spectrum.csv'
out_json = base_dir / f'{spectrum_basename}_real_predicted_params.json'
out_png  = base_dir / f'{spectrum_basename}_spectrum_preview.png'

print(f'Set name       : {SET_NAME or "(legacy)"}')
print(f'runName        : {run_name}')
print(f'spectrum base  : {spectrum_basename}')
print(f'run_dir        : {run_dir}')
print(f'spectrum_csv   : {spectrum_csv}')

if run_name == 'TO_BE_SET_AFTER_FIRST_TRAINING':
    raise RuntimeError(
        'config/inference.json.runName is still the placeholder. '
        'Train a model with 01_train_and_eval.ipynb and update runName.'
    )
if not run_dir.is_dir():
    raise RuntimeError(f'Run directory not found: {run_dir}')
if not spectrum_csv.exists():
    raise RuntimeError(
        f'Spectrum CSV not found: {spectrum_csv}. '
        f'Run matlab/PrepareOneSpectrumForCNN.m to produce it first.'
    )

## 3. Predict

`predict_real_spectrum` builds the 3-channel input using the **same**
`make_three_channel_input` function that training used — no duplication — then loads the
run's model and denormalizes the output back to Gauss (physical units).

In [ ]:
from dyson_cnn.infer import predict_real_spectrum

result = predict_real_spectrum(
    run_dir=run_dir,
    spectrum_csv_path=spectrum_csv,
    out_json_path=out_json,
    out_preview_png_path=out_png,
)

print('\nPredicted parameters (physical units):')
for k, v in result['y_pred_physical'].items():
    print(f'  {k}: {v:.6g}')
print(f'\nSaved JSON: {out_json}')
print(f'Saved PNG : {out_png}')

## 4. Next step

Run [`matlab/Validator.m`](../matlab/Validator.m) to reconstruct the Dysonian curve from
the predicted `B0`, `dB`, `p3` parameters and overlay it on the raw experimental spectrum
for visual quality assessment.